In [1]:
from pathlib import Path
import warnings
from utils.benchmarks import BENCHMARKS_TO_PATHS
warnings.filterwarnings('ignore')

In [2]:
#A program that takes a csv and trains models on it. Streamlined model selection.
#==============================================================================

import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
# import pandas_profiling
import numpy as np
import os
import sys
import traceback
#Fit an xgboost model
import random

#TabNet
# from fast_tabnet.core import *

import shutil

In [3]:
#Project Variables
#===================================================================================================
PROJECT_NAME = 'superstore'
VARIABLE_FILES = False
#Maximum amount of rows to take

## -- STEFANOS -- Remove sample

# SAMPLE_COUNT = 4000
FASTAI_LEARNING_RATE = 1e-1
AUTO_ADJUST_LEARNING_RATE = False
#Set to True automatically infer if variables are categorical or continuous
ENABLE_BREAKPOINT = True
#When trying to declare a column a continuous variable, if it fails, convert it to a categorical variable
CONVERT_TO_CAT = False
REGRESSOR = True
SEP_DOLLAR = False
SEP_COMMA = True
SHUFFLE_DATA = True

In [4]:
import time
start_time = time.time()

In [5]:
### cell 0 ###

benchmark_name = "supermarket-sales-prediction-xgboost-fastai"
PARAM_DIR = Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "superstore"
df = pd.read_csv(Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "superstore" / "SampleSuperstore.csv")
factor = 1000
df = pd.concat([df]*factor)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9994000 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Ship Mode     object 
 1   Segment       object 
 2   Country       object 
 3   City          object 
 4   State         object 
 5   Postal Code   int64  
 6   Region        object 
 7   Category      object 
 8   Sub-Category  object 
 9   Sales         float64
 10  Quantity      int64  
 11  Discount      float64
 12  Profit        float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1.0+ GB


In [6]:
### cell 1 ###

if SEP_DOLLAR:
    #For every column in df_copy, if the column contains a $, make a new column with the value without the $
    for col in df_copy.columns:
        if '$' in df[col].to_string():
            df[col + '_no_dollar'] = df[col].str.replace('$', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df[col + '_no_dollar'] = df[col + '_no_dollar'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')


if SEP_COMMA:
    #For every column in df_copy, if the column contains a %, make a new column with the value without the %
    for col in df.columns:
        if '%' in df[col].to_string() or ',' in df[col].to_string():
            df[col + '_processed'] = df[col].apply(str).str.replace('%', '').str.replace(',', '')
            #Try to convert this new column to a numeric type
            try:
                df[col + '_processed'] = df[col + '_processed'].apply(pd.to_numeric, errors='coerce').dropna()
            except Exception:
                print(f'{col} can not be converted to a float!')

In [7]:
### cell 2 ###

df

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,25.2480,3,0.20,4.1028
9990,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,91.9600,2,0.00,15.6332
9991,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,258.5760,2,0.20,19.3932
9992,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,29.6000,4,0.00,13.3200


In [8]:
### cell 3 ###

df.isna().sum()

Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Sub-Category    0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64

In [9]:
### cell 4 ###

_ = df.select_dtypes(include=['number']).corr()

In [10]:
### cell 5 ###

df.head()

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [11]:
### cell 6 ###

df.describe().T

,count,mean,std,min,25%,50%,75%,max
Postal Code,9994000.0,55190.379428,32062.090767,1040.000,23223.000,56430.5000,90008.000,99301.000
Sales,9994000.0,229.858001,623.213950,0.444,17.280,54.4900,209.940,22638.480
Quantity,9994000.0,3.789574,2.224998,1.000,2.000,3.0000,5.000,14.000
Discount,9994000.0,0.156203,0.206442,0.000,0.000,0.2000,0.200,0.800
Profit,9994000.0,28.656896,234.248399,-6599.978,1.728,8.6665,29.364,8399.976


In [12]:
### cell 7 ###

df.columns

Index(['Ship Mode', 'Segment', 'Country', 'City', 'State', 'Postal Code',
       'Region', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Discount',
       'Profit'],
      dtype='object')

In [13]:
### cell 8 ###

target = ''
target_str = ''
targets = []

#Loop through every possible target column (Continuous)
for i in range(len(df.columns)-1, 0, -1):
    try:
        df[df.columns[i]] = df[df.columns[i]].apply(pd.to_numeric, errors='coerce').dropna()
        target = df.columns[i]
        target_str = target.replace('/', '-')
    except:
        continue
    print(f'Target Variable: {target}')

    #===================================================================================================

    #Create project config files if they don't exist.
    if not os.path.exists(PARAM_DIR):
        #create param_dir
        os.makedirs(PARAM_DIR)
    if not os.path.exists(f'{PARAM_DIR}/cats.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/cats.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/conts.txt'):
        #create param_dir
        with open(f'{PARAM_DIR}/conts.txt', 'w') as f:
            f.write('')
    if not os.path.exists(f'{PARAM_DIR}/cols_to_delete.txt'):
        with open(f'{PARAM_DIR}/cols_to_delete.txt', 'w') as f:
            f.write('')

    df = df.drop_duplicates()
    if SHUFFLE_DATA:
        df = df.sample(frac=1).reset_index(drop=True)

    # workaround for fastai/pytorch bug where bool is treated as object and thus erroring out.
    for n in df:
        if pd.api.types.is_bool_dtype(df[n]):
            df[n] = df[n].astype('uint8')

    with open(f'{PARAM_DIR}/cols_to_delete.txt', 'r') as f:
        cols_to_delete = f.read().splitlines()
    for col in cols_to_delete:
        try:
            del(df[col])
        except:
            pass

    try:
        df = df.fillna(0)
    except:
        pass


    #Auto detect categorical and continuous variables
    #==============================================================================
    likely_cat = {}
    for var in df.columns:
        likely_cat[var] = 1.*df[var].nunique()/df[var].count() < 0.05 #or some other threshold

    cats = [var for var in df.columns if likely_cat[var]]
    conts = [var for var in df.columns if not likely_cat[var]]

    #remove target from lists
    try:
        conts.remove(target)
        cats.remove(target)
    except:
        pass
    #Convert target to float
    df[target] = df[target].apply(pd.to_numeric, errors='coerce').dropna()

    if VARIABLE_FILES == True:
        with open(f'{PARAM_DIR}/cats.txt', 'r') as f:
            cats = f.read().splitlines()

        with open(f'{PARAM_DIR}/conts.txt', 'r') as f:
            conts = f.read().splitlines()

    for var in conts: 
        try:
            df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
        except:
            print(f'Could not convert {var} to float.')
            pass

    if ENABLE_BREAKPOINT == True:
        cont_list = []
        for cont in conts:
            focus_cont = cont
            cont_list.append(cont)
        for var in cont_list:
            try:
                df[var] = df[var].apply(pd.to_numeric, errors='coerce').dropna()
            except:
                print(f'Could not convert {var} to float.')
                cont_list.remove(var)
                if CONVERT_TO_CAT == True:
                    cats.append(var)
                pass

Target Variable: Profit
Target Variable: Discount
Target Variable: Quantity
Target Variable: Sales
Target Variable: Sub-Category
Target Variable: Category
Target Variable: Region
Target Variable: Postal Code
Target Variable: State
Target Variable: City
Target Variable: Country
Target Variable: Segment


In [14]:
### cell 9 ###

df.isna().sum()

Ship Mode       0
Segment         0
Country         0
City            0
State           0
Postal Code     0
Region          0
Category        0
Sub-Category    0
Sales           0
Quantity        0
Discount        0
Profit          0
dtype: int64

In [15]:
end_time = time.time()
print(end_time - start_time)

1572.5027809143066
